In [31]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)

# print(df.shape)
# print(df.head())
# print(df.info())
# print(df.describe())
# print(df.columns.tolist())

# print(df.isnull())
df["target"] = data.target

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

# print(X)
# print(y)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    confusion = confusion_matrix(y_test, predictions)

    results.append({"Model": name,
                    "Accuracy Score": accuracy,
                    "Precision Score": precision,
                    "Recall Score": recall,
                    "F1 Score": f1,
                    "Confusion Matrix": confusion
                   })

results_df = pd.DataFrame(results)
print(results_df)

# The best model is SVM

param_dist = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

model = SVC()

search = RandomizedSearchCV(
     model,
    param_dist,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print(search.best_params_)

best_model = search.best_estimator_

predictions = best_model.predict(X_test)

print(accuracy_score(y_test, predictions))

                 Model  Accuracy Score  Precision Score  Recall Score  \
0  Logistic Regression        0.973684         0.972222      0.985915   
1                  KNN        0.947368         0.957746      0.957746   
2                  SVM        0.982456         0.972603      1.000000   
3        Random Forest        0.964912         0.958904      0.985915   

   F1 Score    Confusion Matrix  
0  0.979021  [[41, 2], [1, 70]]  
1  0.957746  [[40, 3], [3, 68]]  
2  0.986111  [[41, 2], [0, 71]]  
3  0.972222  [[40, 3], [1, 70]]  
{'kernel': 'rbf', 'gamma': 'scale', 'C': 1}
0.9824561403508771
